# Baseline Model

## Table of Contents
1. [Data Import](#Data-Import)
2. [Data Preparation](#Data-Preparation)
3. [Model Choice](#Model-Choice)
4. [Feature Selection](#Feature-Selection)
5. [Implementation](#Implementation)
6. [Evaluation](#Evaluation)

## Data Import

We use the same minimum wage dataset as the rest of the project: county-level annual data from Callaway (2022), covering 2001-2007. The outcome is log teen employment, and the treatment is an indicator for whether the county's minimum wage is above the federal minimum.

In [1]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold
import warnings

warnings.filterwarnings("ignore")
np.random.seed(772023)


In [2]:
data = pd.read_csv("https://raw.githubusercontent.com/CausalAIBook/MetricsMLNotebooks/main/data/minwage_data.csv",
                   index_col=0)
data.head()


,countyreal,state_name,year,FIPS,emp0A01_BS,quarter,censusdiv,pop,annual_avg_pay,state_mw,fed_mw,treated,G,lemp,lpop,lavg_pay,region,ever_treated,id
1,2013,Alaska,2001,2013,15,1,9,2459,22155,5.65,5.15,1,2001,2.708050,7.807510,10.005818,4,1,2013
2,2013,Alaska,2002,2013,17,1,9,2664,28447,5.65,5.15,1,2001,2.833213,7.887584,10.255798,4,1,2013
3,2013,Alaska,2003,2013,12,1,9,2715,30184,7.15,5.15,1,2001,2.484907,7.906547,10.315067,4,1,2013
4,2013,Alaska,2004,2013,13,1,9,2677,27557,7.15,5.15,1,2001,2.564949,7.892452,10.224012,4,1,2013
5,2013,Alaska,2005,2013,11,1,9,2646,30396,7.15,5.15,1,2001,2.397895,7.880804,10.322066,4,1,2013


## Data Preparation

This mirrors the preparation used throughout the project, so the baseline model is evaluated on exactly the same data as the more advanced learners.

We drop counties that were already treated in 2001, drop unused columns, and split counties into treatment (received a minimum wage increase in 2004) and control (stayed at the federal minimum) groups for each year.

In [3]:
data = data.loc[(data.G == 0) | (data.G > 2001)]
data.drop(columns=["countyreal", "state_name", "FIPS", "emp0A01_BS",
                   "quarter", "censusdiv", "pop", "annual_avg_pay",
                   "state_mw", "fed_mw", "ever_treated"], inplace=True)


In [4]:
years = [2001, 2002, 2003, 2004, 2005, 2006, 2007]
treat, cont = {}, {}
for year in years:
    treat[year] = data.loc[(data.G == 2004) & (data.year == year)].copy()
    cont[year] = data.loc[((data.G == 0) | (data.G > year)) & (data.year == year)].copy()


We extract the 2001 baseline control variables (population, average pay, teen employment, region) for both groups, assumed sufficient for the parallel trends assumption to hold.

In [5]:
treat[2001].drop(columns=["year", "G", "region", "treated"], inplace=True)
cont[2001].drop(columns=["year", "G", "region", "treated"], inplace=True)


2003 is the pre-treatment reference period for both groups.

In [6]:
treatB = pd.merge(treat[2003], treat[2001], on="id", suffixes=["_pre", "_0"])
treatB.drop(columns=["treated", "lpop_pre", "lavg_pay_pre", "year", "G"], inplace=True)

contB = pd.merge(cont[2003], cont[2001], on="id", suffixes=["_pre", "_0"])
contB.drop(columns=["treated", "lpop_pre", "lavg_pay_pre", "year", "G"], inplace=True)


For each post-treatment year, we compute the change in log employment (`dy`) relative to 2003, combine treatment and control observations, and one-hot encode region.

In [7]:
tdid, cdid = {}, {}
did_data = {}
for year in [2002, 2004, 2005, 2006, 2007]:
    treat[year].drop(columns=["lpop", "lavg_pay", "year", "G", "region"], inplace=True)
    cont[year].drop(columns=["lpop", "lavg_pay", "year", "G", "region"], inplace=True)

    tdid[year] = pd.merge(treat[year], treatB, on="id")
    tdid[year]["dy"] = tdid[year]["lemp"] - tdid[year]["lemp_pre"]
    tdid[year].drop(columns=["id", "lemp", "lemp_pre"], inplace=True)
    tdid[year].treated = 1

    cdid[year] = pd.merge(cont[year], contB, on="id")
    cdid[year]["dy"] = cdid[year]["lemp"] - cdid[year]["lemp_pre"]
    cdid[year].drop(columns=["id", "lemp", "lemp_pre"], inplace=True)

    did_data[year] = pd.concat((tdid[year], cdid[year]))
    dummy_data = pd.get_dummies(did_data[year].region, drop_first=True, prefix="region")
    did_data[year] = pd.concat((did_data[year], dummy_data), axis=1).drop(columns=["region"])

did_data[2004].head()


,treated,lemp_0,lpop_0,lavg_pay_0,dy,region_2,region_3,region_4
0,1,5.117994,9.776676,10.219356,-0.118482,True,False,False
1,1,6.302619,10.677615,10.505150,0.065813,True,False,False
2,1,7.334329,11.127204,10.162423,0.008202,True,False,False
3,1,3.737670,9.156940,10.283908,-0.336472,True,False,False
4,1,3.970292,8.529912,9.837935,0.125163,True,False,False


## Model Choice

Our baseline is the **"No Controls" doubly robust DiD estimator**. It uses:
- `DummyRegressor(strategy="mean")` as the outcome model, so the predicted change in employment under no treatment is just the sample mean, ignoring all covariates.
- `DummyClassifier(strategy="prior")` as the treatment model, so the predicted treatment probability is just the overall treatment share in the sample, ignoring all covariates.

With no covariates involved, this reduces to the classic, unconditional Difference-in-Differences estimator. We choose it as the baseline because:

1. It is the standard method our project's ML-based estimators (Lasso, Ridge, Random Forest, and the Best/Stack ensembles) are meant to improve on.
2. It requires no feature engineering or model tuning, so it is fully reproducible and easy to defend.
3. It gives a clean reference point: any difference between this estimate and the covariate-adjusted estimates tells us how much the baseline characteristics of counties matter for identifying the treatment effect.

## Feature Selection

The baseline model intentionally uses **no features**. Since the nuisance models are simple means and priors, no covariates are passed to them.

This is different from the other methods used later in the project, which use:
- `lemp_0`, `lpop_0`, `lavg_pay_0` (2001 baseline employment, population, and average pay)
- One-hot encoded region dummies (`region_*`)

Excluding these here is deliberate. It isolates the effect of adding controls, which we evaluate by comparing this baseline against the covariate-adjusted methods later in the project.

## Implementation

We reuse the same doubly robust ATT estimator (`dr_att`) used throughout the project, only with the dummy (no-covariate) nuisance learners.

In [8]:
def final_stage(D, y, Dhat, yhat, phat):
    # doubly robust quantity for every sample
    phihat = ((D - Dhat) / (phat * (1 - Dhat))) * (y - yhat)
    point = np.mean(phihat) / np.mean(D / phat)
    # influence function
    phihat = (phihat - point * (D / phat)) / np.mean(D / phat)
    var = np.mean(np.square(phihat))
    stderr = np.sqrt(var / D.shape[0])
    return point, stderr


In [9]:
def dr_att(X, D, y, modely, modeld, *, trimming=0.01, nfolds=5):
    '''
    Doubly robust DML estimator for the ATT, with cross-fitting.

    X: controls, pandas DataFrame
    D: treatment indicator, numpy array
    y: outcome (dy in DiD), numpy array
    modely: ML model for predicting y under D=0
    modeld: ML model for predicting D
    '''
    cv = StratifiedKFold(n_splits=nfolds, shuffle=True, random_state=1234)
    yhat = np.zeros(y.shape)
    for train, test in cv.split(X, D):
        modely.fit(X.iloc[train][D[train] == 0], y[train][D[train] == 0])
        yhat[test] = modely.predict(X.iloc[test])
    phat = cross_val_predict(DummyRegressor(), X, D, cv=cv)
    Dhat = cross_val_predict(modeld, X, D, cv=cv, method='predict_proba')[:, 1]
    Dhat = np.clip(Dhat, trimming, 1 - trimming)
    point, stderr = final_stage(D, y, Dhat, yhat, phat)
    rmsey = np.sqrt(np.mean((y - yhat)[D == 0]**2))
    rmseD = np.sqrt(np.mean((D - Dhat)**2))
    return point, stderr, yhat, Dhat, rmsey, rmseD, phat


Now we run the baseline (No Controls) estimator for each post-treatment year, 2004 to 2007.

In [10]:
att_baseline, se_baseline, rmsey_baseline, rmseD_baseline = {}, {}, {}, {}

for year in [2004, 2005, 2006, 2007]:
    X = did_data[year].drop(columns=["treated", "dy"])
    D = did_data[year].treated.values
    dy = did_data[year].dy.values

    modely = DummyRegressor(strategy="mean")
    modeld = DummyClassifier(strategy="prior")

    point, stderr, yhat, Dhat, rmsey, rmseD, phat = dr_att(
        X, D, dy, modely, modeld, trimming=0.01, nfolds=5
    )

    att_baseline[year] = point
    se_baseline[year] = stderr
    rmsey_baseline[year] = rmsey
    rmseD_baseline[year] = rmseD

    print(f"Year {year}: ATT = {point:.4f}, SE = {stderr:.4f}")


Year 2004: ATT = -0.0400, SE = 0.0191
Year 2005: ATT = -0.0763, SE = 0.0202
Year 2006: ATT = -0.1168, SE = 0.0198
Year 2007: ATT = -0.1311, SE = 0.0226


## Evaluation

Unlike a standard prediction task, we are not evaluating accuracy or MSE against a held-out test set. Instead, the relevant metrics are:

- **ATT (Average Treatment Effect on the Treated)**: the estimated causal effect of the minimum wage increase on teen employment, for each year.
- **Standard error (SE)** of the ATT estimate, used for inference and confidence intervals.
- **RMSE of the outcome model** (`rmsey`): cross-fitted prediction error of the no-covariate outcome model, evaluated on untreated counties.
- **RMSE of the treatment model** (`rmseD`): cross-fitted prediction error of the no-covariate propensity model.

These numbers serve as the reference point for the rest of the project. Every other method (Lasso, Ridge, Random Forest, Best, Stack) will be compared against this baseline's RMSE (to see whether covariates improve nuisance prediction) and against this baseline's ATT (to see whether covariate adjustment changes the estimated treatment effect).

In [11]:
baseline_table = pd.DataFrame({
    "ATT": att_baseline,
    "SE": se_baseline,
    "RMSE dy": rmsey_baseline,
    "RMSE D": rmseD_baseline,
})
baseline_table


,ATT,SE,RMSE dy,RMSE D
2004,-0.040018,0.019055,0.163240,0.198170
2005,-0.076277,0.020163,0.188136,0.200574
2006,-0.116778,0.019763,0.223408,0.211096
2007,-0.131077,0.022604,0.230234,0.250283
